In [1]:
# Dependencias externas
import pandas as pd

# Dependencias locales
from LabelerV2.modules.matcher import match_columns_optimized

In [3]:
df_to_label = pd.read_csv(r'P:\Programacion\USACH\Investigacion-Redes\nfstream-pcap-to-csv\CSVs\f\USACH\Ayudantias\Investigacion-Redes\PCAPs\Bot-IoT\Theft\Data_Exfiltration\IoT_Dataset_data_theft__00001_20180618110503.csv')

# Transformar a string las columnas clave : "src_ip", "src_port", "dst_ip", "dst_port"
df_to_label["src_ip"] = df_to_label["src_ip"].astype(str)
df_to_label["src_port"] = df_to_label["src_port"].astype(str)
df_to_label["dst_ip"] = df_to_label["dst_ip"].astype(str)
df_to_label["dst_port"] = df_to_label["dst_port"].astype(str)

# Limpiar valores de las columnas clave
key_columns = ["src_ip", "src_port", "dst_ip", "dst_port"]
for column in key_columns:
    df_to_label[column] = df_to_label[column].str.replace(" ", "")
    df_to_label[column] = df_to_label[column].str.replace(".", "")

df_to_label

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,dst2src_fin_packets,application_name,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type
0,0,0,19216810046,be:ff:34:42:10:01,be:ff:34,80,1921681005,ae:bb:10:44:55:60,ae:bb:10,80,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
1,1,0,19216810046,be:ff:34:42:10:01,be:ff:34,80,1921681005,ae:bb:10:44:55:60,ae:bb:10,80,...,0,HTTP,Web,1,1,NaN,NaN,NaN,NaN,NaN
2,2,0,1921681003,00:50:56:be:26:db,00:50:56,60864,17221725138,00:50:56:be:b5:f6,00:50:56,443,...,1,TLS,Web,0,6,NaN,NaN,NaN,NaN,NaN
3,3,0,1921681007,fa:cb:34:12:44:59,fa:cb:34,365,1921681003,ba:ba:ca:6a:00:12,ba:ba:ca,565,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
4,4,0,1921681006,00:50:56:be:4f:b7,00:50:56,138,192168100255,ff:ff:ff:ff:ff:ff,ff:ff:ff,138,...,0,NetBIOS.SMBv1,System,0,6,wininst,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,120,0,192168100150,00:50:56:be:02:54,00:50:56,36874,1921681003,00:50:56:be:26:db,00:50:56,22,...,0,SSH,RemoteAccess,0,6,NaN,905CE75BA9C8343782F8D76D938F5CE8,D41D8CD98F00B204E9800998ECF8427E,NaN,NaN
121,121,0,192168100150,00:50:56:be:02:54,00:50:56,36888,1921681003,00:50:56:be:26:db,00:50:56,22,...,0,SSH,RemoteAccess,0,6,NaN,905CE75BA9C8343782F8D76D938F5CE8,D41D8CD98F00B204E9800998ECF8427E,NaN,NaN
122,122,0,1921681003,00:50:56:be:26:db,00:50:56,50032,192168100150,00:50:56:be:02:54,00:50:56,4433,...,0,Unknown,Unspecified,0,0,NaN,NaN,NaN,NaN,NaN
123,123,0,192168100150,00:50:56:be:02:54,00:50:56,36884,1921681003,00:50:56:be:26:db,00:50:56,22,...,0,SSH,RemoteAccess,0,6,NaN,NaN,NaN,NaN,NaN


In [5]:
reference_df = pd.read_csv(r'P:\Programacion\USACH\Investigacion-Redes\nfstream-pcap-to-csv\CSVs\Labeling\Bot-IoT\Data_exfiltration.csv', sep=';', low_memory=False)

reference_df["saddr"] = reference_df["saddr"].astype(str)
reference_df["sport"] = reference_df["sport"].astype(str)
reference_df["daddr"] = reference_df["daddr"].astype(str)
reference_df["dport"] = reference_df["dport"].astype(str)

key_columns = ["saddr", "sport", "daddr", "dport"]
for column in key_columns:
    reference_df[column] = reference_df[column].str.replace(" ", "")
    if column == "sport" or column == "dport":
        reference_df[column] = reference_df[column].str.replace(".0", "")
    reference_df[column] = reference_df[column].str.replace(".", "")

reference_df

,stime,flgs,proto,saddr,sport,dir,daddr,dport,pkts,bytes,...,dpkts,sbytes,dbytes,rate,srate,drate,record,attack,category,subcategory
0,1.529284e+09,e,udp,1921681006,138,->,192168100255,138,4,986,...,0,986,0,0.002777,0.002777,0.000000,,0,Normal,Normal
1,1.529284e+09,e,tcp,1921681003,60864,<?>,17221725138,443,12,1053,...,4,697,356,0.098173,0.062473,0.026774,,0,Normal,Normal
2,1.529284e+09,e,udp,1921681007,138,->,192168100255,138,4,1086,...,0,1086,0,0.004102,0.004102,0.000000,,0,Normal,Normal
3,1.529284e+09,e,arp,192168100149,nan,who,1921681006,nan,26,1560,...,13,780,780,0.021496,0.010318,0.010318,,1,Theft,Data_Exfiltration
4,1.529284e+09,e,tcp,192168100150,36682,->,1921681003,22,28,5098,...,15,1640,3458,55.147282,24.509903,28.594887,,1,Theft,Data_Exfiltration
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,1.529285e+09,e,arp,1921681004,nan,who,1921681001,nan,2,102,...,1,42,60,3389.830566,0.000000,0.000000,,0,Normal,Normal
138,1.529285e+09,e,arp,192168100150,nan,who,1921681001,nan,2,120,...,1,60,60,8928.571289,0.000000,0.000000,,0,Normal,Normal
139,1.529285e+09,e,udp,1921681005,80,<->,19216810046,3456,15579,15156281,...,7789,7754473,7401808,158.457245,79.228622,79.218452,,0,Normal,Normal
140,1.529285e+09,e,arp,1921681003,nan,who,192168100150,nan,12,720,...,6,360,360,0.036612,0.016642,0.016642,,1,Theft,Data_Exfiltration


In [7]:
# Prueba con merge

# Merge de los DataFrames
merged_df = df_to_label.merge(reference_df,
                              how='left',
                              left_on=["src_ip", "src_port", "dst_ip", "dst_port"],
                              right_on=["saddr", "sport", "daddr", "dport"]
                              )

# Acotar a columnas de referencia
merged_final_columns = df_to_label.columns.tolist() + ["attack", "category", "subcategory"]
#merged_final_columns = ["saddr", "sport", "daddr", "dport"] + ["attack", "category", "subcategory"]
merged_df = merged_df[merged_final_columns]
merged_df

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,attack,category,subcategory
0,0,0,19216810046,be:ff:34:42:10:01,be:ff:34,80,1921681005,ae:bb:10:44:55:60,ae:bb:10,80,...,0,0,NaN,NaN,NaN,NaN,NaN,0,Normal,Normal
1,0,0,19216810046,be:ff:34:42:10:01,be:ff:34,80,1921681005,ae:bb:10:44:55:60,ae:bb:10,80,...,0,0,NaN,NaN,NaN,NaN,NaN,0,Normal,Normal
2,1,0,19216810046,be:ff:34:42:10:01,be:ff:34,80,1921681005,ae:bb:10:44:55:60,ae:bb:10,80,...,1,1,NaN,NaN,NaN,NaN,NaN,0,Normal,Normal
3,1,0,19216810046,be:ff:34:42:10:01,be:ff:34,80,1921681005,ae:bb:10:44:55:60,ae:bb:10,80,...,1,1,NaN,NaN,NaN,NaN,NaN,0,Normal,Normal
4,2,0,1921681003,00:50:56:be:26:db,00:50:56,60864,17221725138,00:50:56:be:b5:f6,00:50:56,443,...,0,6,NaN,NaN,NaN,NaN,NaN,0,Normal,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,120,0,192168100150,00:50:56:be:02:54,00:50:56,36874,1921681003,00:50:56:be:26:db,00:50:56,22,...,0,6,NaN,905CE75BA9C8343782F8D76D938F5CE8,D41D8CD98F00B204E9800998ECF8427E,NaN,NaN,1,Theft,Data_Exfiltration
123,121,0,192168100150,00:50:56:be:02:54,00:50:56,36888,1921681003,00:50:56:be:26:db,00:50:56,22,...,0,6,NaN,905CE75BA9C8343782F8D76D938F5CE8,D41D8CD98F00B204E9800998ECF8427E,NaN,NaN,1,Theft,Data_Exfiltration
124,122,0,1921681003,00:50:56:be:26:db,00:50:56,50032,192168100150,00:50:56:be:02:54,00:50:56,4433,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Data_Exfiltration
125,123,0,192168100150,00:50:56:be:02:54,00:50:56,36884,1921681003,00:50:56:be:26:db,00:50:56,22,...,0,6,NaN,NaN,NaN,NaN,NaN,1,Theft,Data_Exfiltration


In [66]:
# Prueba con merge V2

# Renombrar columnas del df_to_label
df_to_label = df_to_label.rename(columns={"src_ip": "saddr", "src_port": "sport", "dst_ip": "daddr", "dst_port": "dport"})

# Merge de los DataFrames
merged_df = df_to_label.merge(reference_df,
                              how='left',
                              on=["saddr", "sport", "daddr", "dport"],
                              indicator=True,
                              )

# Acotar a columnas de referencia
#merged_final_columns = df_to_label.columns.tolist() + ["attack", "category", "subcategory"]
merged_final_columns = ["saddr", "sport", "daddr", "dport"] + ["attack", "category", "subcategory"]
merged_df = merged_df[merged_final_columns]
merged_df


,saddr,sport,daddr,dport,attack,category,subcategory
0,192.168.100.46,80,192.168.100.5,80,NaN,NaN,NaN
1,192.168.100.46,80,192.168.100.5,80,NaN,NaN,NaN
2,192.168.100.3,60864,172.217.25.138,443,NaN,NaN,NaN
3,192.168.100.7,365,192.168.100.3,565,NaN,NaN,NaN
4,192.168.100.6,138,192.168.100.255,138,NaN,NaN,NaN
...,...,...,...,...,...,...,...
120,192.168.100.150,36874,192.168.100.3,22,NaN,NaN,NaN
121,192.168.100.150,36888,192.168.100.3,22,NaN,NaN,NaN
122,192.168.100.3,50032,192.168.100.150,4433,NaN,NaN,NaN
123,192.168.100.150,36884,192.168.100.3,22,NaN,NaN,NaN


In [48]:
# Prueba con concat

# Concatenar df_to_label con df_reference
concated_df = pd.concat([df_to_label,reference_df], join='inner', axis=1)

concated_final_columns = df_to_label.columns.tolist() + ["attack", "category", "subcategory"]
#concated_final_columns = ["saddr", "sport", "daddr", "dport"] + ["attack", "category", "subcategory"]
concated_df = concated_df[concated_final_columns]
concated_df

,id,expiration_id,saddr,saddr,src_mac,src_oui,sport,sport,daddr,daddr,...,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,attack,category,subcategory
0,0,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,42954,54766,192.168.100.150,192.168.100.149,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
1,1,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,54770,42954,192.168.100.149,192.168.100.150,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
2,2,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,42964,54768,192.168.100.150,192.168.100.149,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
3,3,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,54780,42956,192.168.100.149,192.168.100.150,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
4,4,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,42974,54770,192.168.100.150,192.168.100.149,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
785,785,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,43372,43378,192.168.100.150,192.168.100.150,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
786,786,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,55188,55192,192.168.100.149,192.168.100.149,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
787,787,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,43382,43380,192.168.100.150,192.168.100.150,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging
788,788,0,192.168.100.3,192.168.100.3,00:50:56:be:26:db,00:50:56,55198,55194,192.168.100.149,192.168.100.149,...,0,0,NaN,NaN,NaN,NaN,NaN,1,Theft,Keylogging


In [39]:
# Prueba con join

# Renombrar columnas del df_to_label
df_to_label = df_to_label.rename(columns={"src_ip": "saddr", "src_port": "sport", "dst_ip": "daddr", "dst_port": "dport"})

# Join df_to_label con df_reference por los índices
joined_df = df_to_label.join(reference_df.set_index(["saddr", "sport", "daddr", "dport"]),
                             on=["saddr", "sport", "daddr", "dport"],
                             how='left')


#joined_final_columns = df_to_label.columns.tolist() + ["attack", "category", "subcategory"]
joined_final_columns = ["saddr", "sport", "daddr", "dport"] + ["attack", "category", "subcategory"]
joined_df = joined_df[joined_final_columns]
joined_df

,saddr,sport,daddr,dport,attack,category,subcategory
0,192.168.100.3,42954,192.168.100.150,4433,1.0,Theft,Keylogging
1,192.168.100.3,54770,192.168.100.149,4433,1.0,Theft,Keylogging
2,192.168.100.3,42964,192.168.100.150,4433,1.0,Theft,Keylogging
3,192.168.100.3,54780,192.168.100.149,4433,1.0,Theft,Keylogging
4,192.168.100.3,42974,192.168.100.150,4433,1.0,Theft,Keylogging
...,...,...,...,...,...,...,...
785,192.168.100.3,43372,192.168.100.150,4433,1.0,Theft,Keylogging
786,192.168.100.3,55188,192.168.100.149,4433,1.0,Theft,Keylogging
787,192.168.100.3,43382,192.168.100.150,4433,1.0,Theft,Keylogging
788,192.168.100.3,55198,192.168.100.149,4433,1.0,Theft,Keylogging
